In [21]:
import numpy as np 
from vosk import KaldiRecognizer , Model
import soundfile as sf 
import shutil
import subprocess
import librosa
from pathlib import Path
from typing import Tuple
from faster_whisper import WhisperModel
import unicodedata 
import re
import contractions
from num2words import num2words
import difflib
from itertools import zip_longest
from glob import glob
import time
import json

# Audio Pipeline and Standardization

The first step in our evaluation process is to ensure all audio input is standardized. Variations in file formats, sample rates, and channel counts can lead to inconsistent model performance. To prevent this, we developed a preprocessing pipeline using **FFmpeg** to normalize all incoming audio.

### Audio Conversion Strategy
We chose the **MP3** format with specific parameters optimized for the ASR engines:
* **Sample Rate (16kHz):** This is the industry standard for speech recognition; it captures all necessary vocal frequencies while reducing file size.
* **Mono Channel:** Since we are transcribing individual speech, converting stereo audio to mono simplifies the processing and reduces CPU load.
* **High-Quality VBR:** We use a high-quality variable bitrate (qscale:a 2) to ensure no phonetic data is lost during compression.



In [2]:
def ensure_ffmpeg_available():
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found on PATH. Install it in the container.")
        
def convert_to_mp3(in_path: str, out_path: str, timeout: int = 60) -> Tuple[str, str]:
    ensure_ffmpeg_available()
    in_path = str(Path(in_path))
    out_path = str(Path(out_path))
    
    cmd = [
        "ffmpeg", "-y",
        "-nostdin",
        "-v", "warning",          
        "-i", in_path,
        "-codec:a", "libmp3lame", # MP3 Encoder
        "-qscale:a", "2",          # High quality variable bitrate
        "-ar", "16000",            # 16kHz (Standard for Whisper)
        "-ac", "1",                # Mono (Standard for Whisper)
        out_path
    ]
    
    proc = subprocess.run(cmd, capture_output=True, text=True, check=False, timeout=timeout)
    if proc.returncode != 0:
        raise RuntimeError(f"ffmpeg failed (rc={proc.returncode}): {proc.stderr.strip()}")
    
    return proc.stdout, proc.stderr

# Data Collection

To ensure our evaluation reflected real-world conditions, we created a custom dataset of recordings from non-native children at home. These samples allowed us to test how each model handles the specific high-pitched frequencies and phonetic patterns of children's speech in typical household acoustic environments. This original data served as our primary benchmark for identifying the most reliable model for actual production use.

In [3]:
test_data = glob('/home/waad/Documents/voice_recognition/final_test_audios/audio*.wav')
j=1
for i in test_data:
    stdout, stderr =  convert_to_mp3(i, f"test_audio/audio{j}.mp3", timeout=30)
    j+=1

# Signal Quality Assessment (SNR)

Before transcription, we perform a Signal-to-Noise Ratio (SNR) analysis to assess the acoustic quality of the recording. This step allows the system to make an automated **denoising decision**, ensuring that noisy audio is pre-processed before it reaches an ASR engine that is not natively robust to high-interference environments.

In [27]:
def estimate_snr(audio_path, silence_threshold_db=-40):
    audio, sr = sf.read(audio_path)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    frame_size = int(0.02 * sr)   
    hop_size = frame_size

    rms_values = []
    for i in range(0, len(audio) - frame_size, hop_size):
        frame = audio[i:i+frame_size]
        rms = np.sqrt(np.mean(frame**2) + 1e-10)
        rms_db = 20 * np.log10(rms + 1e-10)
        rms_values.append(rms_db)

    rms_values = np.array(rms_values)

  
    silence_frames = rms_values < silence_threshold_db
    speech_frames = rms_values >= silence_threshold_db

    if silence_frames.sum() < 5 or speech_frames.sum() < 5:
        return None 

    noise_level = np.mean(rms_values[silence_frames])
    speech_level = np.mean(rms_values[speech_frames])

    snr = speech_level - noise_level
    return snr


def denoise_decision(snr):
    if snr is None:
        denoise = False
    elif snr < 20:
        denoise = True
    else:
        denoise = False
    return denoise


In [5]:
test_data = glob('test_audio/audio*.mp3')

In [28]:
from IPython.display import Audio , display
count =0 
for i in test_data :
     snr = estimate_snr(i)
     denoise = denoise_decision(snr)
    # display(Audio(i))
     print(f"SNR: {snr} dB, Denoise: {denoise}")
     if denoise == True:
         count+=1
print("\n")
print(count)

SNR: 29.422576419460672 dB, Denoise: False
SNR: 37.203764183572744 dB, Denoise: False
SNR: 48.54889777809072 dB, Denoise: False
SNR: 36.09383143373469 dB, Denoise: False
SNR: 56.80612038834974 dB, Denoise: False
SNR: 37.321955705025985 dB, Denoise: False
SNR: 35.39485216356684 dB, Denoise: False
SNR: 44.41016194622196 dB, Denoise: False
SNR: 35.61279057430878 dB, Denoise: False
SNR: 52.694438529152684 dB, Denoise: False
SNR: 28.57479866687418 dB, Denoise: False
SNR: 37.818694123128196 dB, Denoise: False
SNR: 41.22036875135869 dB, Denoise: False
SNR: 34.55679588923518 dB, Denoise: False
SNR: 37.18552730445214 dB, Denoise: False
SNR: 32.759148243108264 dB, Denoise: False
SNR: 32.45140955294792 dB, Denoise: False
SNR: 40.57909346265523 dB, Denoise: False
SNR: 34.53250026916608 dB, Denoise: False
SNR: 37.918992670766045 dB, Denoise: False
SNR: 33.337393749556455 dB, Denoise: False
SNR: 39.14312631825692 dB, Denoise: False
SNR: 31.907128275353536 dB, Denoise: False
SNR: 33.85621133274242 dB

# Model Selection Matrix

### Objective  
Deploy a **resource-optimized, CPU-efficient ASR model** capable of maintaining **high transcription accuracy** for **non-native children’s speech** in real-world environments.

| Evaluation Metric | **Vosk (Kaldi)** | **OpenAI Whisper** | **Faster-Whisper** |
|------------------|-----------------|-------------------|-------------------|
| **Transcription Accuracy** | Moderate | Excellent | Excellent |
| **Computational Weight** | Lightweight | Heavy | Balanced |
| **Latency** | Fast | High (especially on CPU) | Moderate |
| **CPU Utilization** | Very Low | Very High | Moderate |
| **GPU Dependency** | CPU-only | GPU-accelerated (CPU supported) | CPU-only (GPU optional) |
| **Noise Robustness** | Basic | High | High |
| **Children’s Speech Performance** | Moderate | High  | High  |
| **Non-Native Accent Handling** | Poor | Excellent | Excellent |
| **Silence Handling** | Limited | Implicit (model-based) | Explicit VAD integration |
| **Resource Consumption** | Extremely Efficient | Resource-Intensive | Balanced |

### **Final Verdict**

- **Faster-Whisper** provides the best trade-off between **high transcription accuracy** and **computational efficiency**, making it the most suitable choice for CPU-based deployment.
- **Vosk** offers **minimal resource consumption** and **real-time, low-latency transcription**; however, its performance on **non-native children’s speech** is limited.
- **OpenAI Whisper** achieves **excellent transcription accuracy** but incurs **significant latency and CPU overhead** in **CPU-only deployments**, reducing its practicality for real-time applications.

### **Experimental Scope Decision**

Based on the evaluation criteria and the comparative analysis presented above, the experimental phase will be limited to testing **multiple configurations of Vosk and Faster-Whisper models** on the collected dataset.  
This decision was driven by their suitability for **CPU-based deployment**, **real-time inference**, and **resource-constrained environments**.  
OpenAI Whisper was excluded from extensive testing due to its **high computational cost and latency** in CPU-only settings.


### **Ground Truth Establishment**

Prior to conducting model evaluation, we established a **Ground Truth** targets consisting of manual, human-verified transcriptions for all test audio samples. This represents the definitive reality of what a human hears, accounting for the unique phonetic variabilities of non-native pediatric speech. By using these human-generated targets as a fixed reference point, we were able to objectively compare each model's transcription against a 100% accurate baseline, ensuring our final model selection is based on quantifiable precision rather than subjective observation.

In [8]:
targets = [
    "august",
    "may",
    "July",
    "November",
    "February",
    "Have three pencils.",
    "I love my friends.",
    "The baby is sleeping",
    "She is reading a story",
    "I love my father.",
    "It is a square.",
    "It is a circle. ...",
    "I love Strawberry. ...",
    "six years old.",
    "I don't like broccoli",
    "nice to meet you!",
    "This is a circle.",
    "I like pizza.",
    "It is a square.",
    "I like the color blue.",
    "1, 2, 3, 4",
    "The dog is running",
    "The sky is blue",
    "The sun is yellow.",
    "The ball is red.",
    "Red car",
    "one plus one is two",
    "A square.",
    "One, two, three, four, five.",
    "It's the dog!",
    "This is my mom.",
    "I love my mom.",
    "Good morning, good night.",
    "Strawberry are red.",
    "Sunday, Monday, Tuesday, Wednesday, Thursday, Friday",
    "The cat is under the",
    "I love my family.",
    "It is sunny today.",
    "Nice to meet you.",
    "There are 4 cats",
    "She is sitting down.",
    "I am fine, thank you.",
    "It is a square.",
    "1, 2, 3, 4, 5",
    "I am 6 years old",
    "The cat is under the chair.",
    "I like the color blue.",
    "I love pizza!",
    "He is reading his",
    "She is reading a story.",
    "the is sleeping now",
    "3 pencils ....",
    "I have one bag.",
    "I have one",
    "This is my mom",
    "Thank my mom",
    "There are two bags.",
    "Their are welcome.",
    "Have a nice day!",
    "Thank you very much.",
    "my dad",
    "There are two dogs.",
    "You are welcome.",
    "He is jumping high.",
    "I am  playing now!",
    "He is reading a book.",
    "Hello, how are you?",
    "I am Eating an apple.",
    "I am drinking water",
    "The rabbit is small.",
    "The sun is yellow.",
    "The book is on the table.",
    "The dog is running",
    "Apple is red.",
    "The cow is big!",
    "The fish is in the water.",
    "The shoes are in the box.",
    "The boy is at school.",
    "The bird is fly",
    "The is green",
    "The sky is blue.",
    "The ball is red.",
    "Mangoes are orange.",
    "They are 4 cats.",
    "Grass green",
    "I am fine, thank you!",
    "Have a nice day!",
    "Reading a book.",
    "Pink Jacket",
    "He touch grass.",
    "The cat is on the tree.",
    "It's sunny today.",
    "I have one bag.",
    "Hello, how are you?",
    "The ball is red.",
    "Seven days a week.",
    "I don't like broccoli.",
    "I am six.",
    "I like red.",
    "A square.",
    "Four apples.",
    "In the forest.",
    "The cow is big.",
    "1, 2, 3, 4, 5",
    "I like the color blue",
    "Monday, Tuesday, Wednesday, Thursday",
    "I have three pencils",
    "There are two dogs",
    "This is my first time.",
    "I can count to 10",
    "my family is nice",
    "I have a brother",
    "I have a sister.",
    "See you later!",
    "this is my dad",
    "This is my mom.",
    "Hello, how are you?",
    "Thank you very much.",
    "I am playing now",
    "I am drinking water",
    "Have a nice day!",
    "The boy is at school.",
    "The Apple is red.",
    "I am eating an apple",
    "The dog is running",
    "The book is on the table.",
    "The girl is at home.",
    "The fish is in the water.",
    "the toy is next to the bed",
    "the shoes are in the box",
    "it is sunny today",
    "The bird can fly.",
    "The grass is green",
    "The sky is blue.",
    "The sun is yellow.",
    "The cow is big!",
    "The rabbit is small.",
    "The cat is black.",
    "Nice to meet you!",
    "I am fine. Thank you",
    "I like strawberries",
    "There are four cats.",
    "I love pizza.",
    "I am 6 years old.",
    "Good morning. Good night.",
    "mangoes are orange",
    "It's a square.",
    "He is brushing his hair.",
    "She is sitting down.",
    "This is a circle.",
    "The cat is under the chair."
]

# Text Normalization and Standardization

To ensure a fair and scientifically accurate comparison between the **Human Ground Truth** and the **Model Hypothesis**, we implemented a comprehensive text-preprocessing pipeline. This standardization prevents the Word Error Rate (WER) from being artificially inflated by stylistic or formatting differences.

### The Normalization Strategy
The goal is to reduce both text strings to their most basic verbal form: **lowercase, space-separated words with no punctuation or numerical digits.**

In [7]:

def remove_accents(text):
    nfkd_form = unicodedata.normalize('NFKD', text)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])


def text_preprocessing(text):
    text = text.lower().strip()
    text = contractions.fix(text)
    text = remove_accents(text)
    def convert_decimal(match):
        number = match.group(0)
        integer, decimal = number.split(".")
        integer_words = num2words(int(integer))
        decimal_words = num2words(int(decimal))
        return f"{integer_words} point {decimal_words}"
        
    text = re.sub(r"\b\d+\.\d+\b", convert_decimal, text)
    text = re.sub(r"\b\d+\b", lambda x: num2words(int(x.group(0))), text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.split()

# Models Selected for Testing

After preparing our data, we chose five different models to test. We wanted to find a model that is fast on a CPU but still smart enough to understand children's voices. 

We are testing two different "engines" (Faster-Whisper and Vosk) at different sizes to see which one works best.

### The Models We Are Testing:

| Model Engine | Model Size | Strength |
| :--- | :--- | :--- |
| **Faster-Whisper** | **Tiny** | Maximum throughput; lowest CPU overhead. |
| **Faster-Whisper** | **Base** | Balanced speed for real-time applications. |
| **Faster-Whisper** | **Small** | Improved linguistic context for children's speech. |
| **Vosk** | **Small** | Very lightweight; ideal for constrained edge CPUs. |
| **Vosk** | **en-us-0.22** | High performance offline transcription without Transformers. |

In [9]:
small = WhisperModel("small.en", device="cpu", compute_type="int8")

In [10]:
base = WhisperModel("base.en", device="cpu", compute_type="int8")

In [11]:
tiny = WhisperModel("tiny.en", device="cpu", compute_type="int8")

In [12]:
model_path = "/home/waad/Documents/voice_recognition/vosk-model-en-us-0.22"
vosk_us = Model(model_path)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=13 max-active=7000 lattice-beam=6
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:11:12:13:14:15
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from /home/waad/Documents/voice_recognition/vosk-model-en-us-0.22/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:279) Loading HCLG from /home/waad/Documents/voice_recognition/vosk-model-en-us-0.22/graph/HCLG.fst
LOG (VoskAPI:ReadDataFiles():model.cc:297) Loading words from /home/waad/Documents/voice_recognition/vosk-model-en-us-0.22/graph/words.txt
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading wi

In [13]:
model_path = "/home/waad/Documents/voice_recognition/vosk-model-small-en-us-0.15"
vosk_small = Model(model_path)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from /home/waad/Documents/voice_recognition/vosk-model-small-en-us-0.15/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from /home/waad/Documents/voice_recognition/vosk-model-small-en-us-0.15/graph/HCLr.fst /home/waad/Documents/voice_recognition/vosk-model-small-en-us-0.15/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo /home/waad/Documents/voice_recognition/vos

In [14]:
us_recognizer = KaldiRecognizer(vosk_us,16000)
us_recognizer.SetWords(True)

In [15]:
small_recognizer = KaldiRecognizer(vosk_small,16000)
small_recognizer.SetWords(True)

# Implementation of the Models

To get the transcription from our audio files, we created two main functions. These functions take an audio file as input and return the transcribed text using either the **Vosk** or **Faster-Whisper** engine.

### 1. Vosk Transcription 
Vosk is designed to work with raw audio data. We use **librosa** to load the audio and convert it into a format that the Vosk recognizer can understand.


In [16]:
def audio_test_vosk(audio_path,recognizer):
    noisy_audio,sampling_rate = librosa.load(audio_path, sr= 16000, mono = True)
    raw_noisy_audio = (noisy_audio*32767).astype(np.int16).tobytes()
    recognizer.AcceptWaveform(raw_noisy_audio)
    final_result = json.loads(recognizer.FinalResult())
    full_transcript = final_result.get("text","")
    return full_transcript

### 2. Faster-Whisper Transcription 
Faster-Whisper is much more direct. It takes the audio path and uses the "Beam Search" method to improve accuracy by checking multiple possible word sequences.

In [17]:
def audio_test_whisper(file_path, model):
        segments, info = model.transcribe(file_path, beam_size=5)
        text = " ".join([segment.text for segment in segments])
        return text 

# Model Evaluation and Sequence Matching

After cleaning the text, we need a way to compare the model’s transcription to the human ground truth. We used a process called **Sequence Matching** to look at every single word and determine if the model was correct, missed a word, or made a mistake.

### How the Scoring Works
We developed a custom evaluation function that classifies every word into one of four categories:

* **Match:** The model heard the word perfectly.
* **Missed:** The model skipped a word that the human heard.
* **Accepted Typo:** The model was very close (70% similarity or higher). This is helpful for children's speech where a slight misspelling might still be the correct word.
* **Wrong Word:** The model heard something completely different.


In [19]:
## sequance matching 

def sequance_matching_score(target_tokens, vosk_tokens):

    matcher = difflib.SequenceMatcher(None, target_tokens, vosk_tokens)
    
    report = []
    correct_words_count = 0
    total_teacher_words = len(target_tokens)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        
        if tag == 'equal':
            chunk_len = i2 - i1
            correct_words_count += chunk_len
            
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "match"
                })

        elif tag == 'delete':
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "Missed"
                })

        elif tag == 'replace':

            teacher_chunk = target_tokens[i1:i2]
            vosk_chunk = vosk_tokens[j1:j2]
            
            for t_word, v_word in zip_longest(teacher_chunk, vosk_chunk, fillvalue=None):
                
                if t_word is None:
                    break  

                if v_word is None:
                    report.append({
                        "word": t_word,
                        "status": "Missed"
                    })
                    continue 

                else:
                    similarity = difflib.SequenceMatcher(None, t_word, v_word).ratio()
                    
                    if similarity >= 0.7:
                        correct_words_count += 1
                        report.append({
                            "word": t_word, 
                            "status": f"Accepted typo ({int(similarity*100)}%)"
                        })
                    else:
                        report.append({
                            "word": t_word, 
                            "status": f"Wrong word. Heard '{v_word}'"
                        })

    if total_teacher_words == 0:
        final_score = 0
    else:
        final_score = int((correct_words_count / total_teacher_words) * 100)

    return final_score, report


# Model Testing

After implementing the models and evaluation logic, we executed a comprehensive test across our human-verified targets. Each audio sample was processed by all five model configurations to measure real-world performance.

### Testing Workflow
The evaluation process followed an iterative loop for every test sample:
1. **Transcription:** The raw audio was transcribed by each model.
2. **Timing:** Latency was measured using `time.time()` to calculate the speed of each inference.
3. **Normalization:** Both the target (human) and transcribed (model) texts were cleaned using our standardization pipeline.
4. **Scoring:** The **Sequence Matching** algorithm generated a word-by-word comparison report and a final accuracy score.

In [22]:

tiny_acc_shadwing = []
tiny_latency = [] 

base_acc_shadwing = []
base_latency = [] 

small_acc_shadwing = []
small_latency = [] 

vosk_us_acc_shadwing = []
vosk_us_latency = [] 

vosk_small_acc_shadwing = []
vosk_small_latency = [] 


for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test_whisper(i, tiny)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test_whisper(i, base)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test_whisper(i, small)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)

    # Test Vosk_us
    start = time.time()
    raw_vosk_us = audio_test_vosk(i, us_recognizer)
    dur_vosk_us = time.time() - start
    vosk_us_latency.append(dur_vosk_us)
    vosk_us_norm = text_preprocessing(raw_vosk_us)

    # Test Vosk_small
    start = time.time()
    raw_vosk_small = audio_test_vosk(i, small_recognizer)
    dur_vosk_small = time.time() - start
    vosk_small_latency.append(dur_vosk_small)
    vosk_small_norm = text_preprocessing(raw_vosk_small)
    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print(f"Vosk us:  {vosk_us_norm}  (Time: {dur_vosk_us:.3f}s)")
    print(f"Vosk Small: {vosk_small_norm}  (Time: {dur_vosk_small:.3f}s)")
    print("\n\n")
    
    score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
    score_base, report_base = sequance_matching_score(target_norm, base_norm)
    score_small, report_small = sequance_matching_score(target_norm, small_norm)
    score_vosk_us, report_vosk_us = sequance_matching_score(target_norm, vosk_us_norm)
    score_vosk_small, report_vosk_small = sequance_matching_score(target_norm, vosk_small_norm)

    tiny_acc_shadwing.append(score_tiny)
    base_acc_shadwing.append(score_base)
    small_acc_shadwing.append(score_small)
    vosk_us_acc_shadwing.append(score_vosk_us)
    vosk_small_acc_shadwing.append(score_vosk_small)


    print("--- TINY REPORT ---")
    for item in report_tiny:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- BASE REPORT ---")
    for item in report_base:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- SMALL REPORT ---")
    for item in report_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")

    print("--- VOSK US REPORT ---")
    for item in report_vosk_us:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")

    print("--- VOSK SMALL REPORT ---")
    for item in report_vosk_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    
    print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small} | Vosk us: {score_vosk_us}  | Vosk Small: {score_vosk_small}")
    
    print("\n\n")
    print("__________________________________________________________________")



# Shadowing Pass Rate (%)
threshold = 70
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]
shadow_Vosk_us_pass = [score >= threshold for score in vosk_us_acc_shadwing]
shadow_Vosk_small_pass = [score >= threshold for score in vosk_small_acc_shadwing]

shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100
shadow_Vosk_us_acc = sum(shadow_Vosk_us_pass) / len(shadow_Vosk_us_pass) * 100
shadow_Vosk_small_acc = sum(shadow_Vosk_small_pass) / len(shadow_Vosk_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)
shadow_Vosk_us_avg = sum(vosk_us_acc_shadwing) / len(vosk_us_acc_shadwing)
shadow_Vosk_small_avg = sum(vosk_small_acc_shadwing) / len(vosk_small_acc_shadwing)
# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)
avg_time_vosk_us = sum(vosk_us_latency) / len(vosk_us_latency)
avg_time_vosk_small = sum(vosk_small_latency) / len(vosk_small_latency)

print("\n=== FINAL RESULTS ===")

print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")
print(f"Shadowing Accuracy Vosk US:    {shadow_Vosk_us_acc:.2f}%")
print(f"Shadowing Accuracy Vosk Small: {shadow_Vosk_small_acc:.2f}%")

print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")
print(f"Shadowing Avg Score - Vosk US:    {shadow_Vosk_us_avg:.2f}")
print(f"Shadowing Avg Score - Vosk Small: {shadow_Vosk_small_avg:.2f}")
print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")
print(f"Avg Latency (sec) - Vosk US:    {avg_time_vosk_us:.4f}s")
print(f"Avg Latency (sec) - Vosk Small: {avg_time_vosk_small:.4f}s")

Target:        ['august']
Whisper Tiny:  ['oh', 'baby']  (Time: 2.015s)
Whisper Base:  ['oh', 'guys']  (Time: 1.373s)
Whisper Small: ['oh', 'good']  (Time: 10.903s)
Vosk us:  ['all', 'goods', 'all', 'goods']  (Time: 1.204s)
Vosk Small: ['oh', 'guys']  (Time: 0.536s)



--- TINY REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- BASE REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- VOSK US REPORT ---
august          | Wrong word. Heard 'all'
_____________________________________
--- VOSK SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
Scores -> Tiny: 0 | Base: 0 | Small: 0 | Vosk us: 0  | Vosk Small: 0



__________________________________________________________________
Target:        ['may']
Whisper Tiny:  ['me']  (Time: 2.249s)
Whisper Base:  ['may']  

# Parameter Tuning

Having selected the core engine, the project has moved into the **Tuning Phase**. In this stage, we are not just testing different models, but rather "fine-tuning" the internal settings of the **Faster-Whisper** architecture. 

### Key Tuning Parameters Under Evaluation

We are moving beyond default settings to find the optimal configuration for our specific use case. The focus is on balancing transcription quality with CPU performance.

* **Beam Size Tuning**: We are testing 2 different values of beam sizes **5 (industry standard)** and **8 (more accurate)**. While a higher beam size improves accuracy for difficult accents, it significantly increases CPU load and latency. By comparing 5 and 8, we are checking if the extra accuracy provided by a Beam Size of 8 is worth the increased processing time. If the **Average Accuracy Score** does not increase significantly at 8, we will finalize the API with a Beam Size of 5 to ensure the fastest possible user experience.

* **Conditioning on Previous Text**: By default, Whisper models use previous context to guess the next word. However, for children's speech, this often triggers **"hallucination loops"** where the model repeats phrases. We are testing if setting this to **False** results in cleaner, more independent transcripts.

* **VAD Sensitivity**: We are refining how the model filters silence and background noise. This ensures the CPU only processes segments containing actual speech, preventing the model from trying to "transcribe" background static.

In [23]:
def audio_test_tunned_whisper(file_path, model, beam_size=5, condition=False):
        segments, info = model.transcribe(file_path,
                                          language="en",
                                          vad_filter=True,
                                          beam_size=beam_size,
                                          condition_on_previous_text=condition)
        text = " ".join([segment.text for segment in segments])
        return text 

**Beam Size** = `5`  
**VAD Filter** = `True`  
**condition on previous text** =  `False`  

In [24]:

tiny_acc_shadwing = []
tiny_latency = [] 

base_acc_shadwing = []
base_latency = [] 

small_acc_shadwing = []
small_latency = [] 

vosk_us_acc_shadwing = []
vosk_us_latency = [] 

vosk_small_acc_shadwing = []
vosk_small_latency = [] 


for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test_tunned_whisper(i, tiny)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test_tunned_whisper(i, base)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test_tunned_whisper(i, small)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)

    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print("\n\n")
    
    score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
    score_base, report_base = sequance_matching_score(target_norm, base_norm)
    score_small, report_small = sequance_matching_score(target_norm, small_norm)
    
    tiny_acc_shadwing.append(score_tiny)
    base_acc_shadwing.append(score_base)
    small_acc_shadwing.append(score_small)


    print("--- TINY REPORT ---")
    for item in report_tiny:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- BASE REPORT ---")
    for item in report_base:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- SMALL REPORT ---")
    for item in report_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    
    print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small} ")
    
    print("\n\n")
    print("__________________________________________________________________")



# Shadowing Pass Rate (%)
threshold = 70
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]


shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)

# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)

print("\n=== FINAL RESULTS ===")

print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")

print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")

print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")


Target:        ['august']
Whisper Tiny:  ['oh', 'baby']  (Time: 1.658s)
Whisper Base:  ['oh', 'bless']  (Time: 1.062s)
Whisper Small: ['oh', 'good']  (Time: 9.275s)



--- TINY REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- BASE REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
Scores -> Tiny: 0 | Base: 0 | Small: 0 



__________________________________________________________________
Target:        ['may']
Whisper Tiny:  ['me']  (Time: 1.638s)
Whisper Base:  ['may']  (Time: 1.067s)
Whisper Small: ['me']  (Time: 3.007s)



--- TINY REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
--- BASE REPORT ---
may             | match
_____________________________________
--- SMALL REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
Sc

**Beam Size** = `8`  
**VAD Filter** = `True`  
**condition on previous text** =  `False`  

In [26]:

tiny_acc_shadwing = []
tiny_latency = [] 

base_acc_shadwing = []
base_latency = [] 

small_acc_shadwing = []
small_latency = [] 

vosk_us_acc_shadwing = []
vosk_us_latency = [] 

vosk_small_acc_shadwing = []
vosk_small_latency = [] 


for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test_tunned_whisper(i, tiny,8)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test_tunned_whisper(i, base,8)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test_tunned_whisper(i, small,8)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)

    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print("\n\n")
    
    score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
    score_base, report_base = sequance_matching_score(target_norm, base_norm)
    score_small, report_small = sequance_matching_score(target_norm, small_norm)
    
    tiny_acc_shadwing.append(score_tiny)
    base_acc_shadwing.append(score_base)
    small_acc_shadwing.append(score_small)


    print("--- TINY REPORT ---")
    for item in report_tiny:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- BASE REPORT ---")
    for item in report_base:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- SMALL REPORT ---")
    for item in report_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    
    print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small} ")
    
    print("\n\n")
    print("__________________________________________________________________")



# Shadowing Pass Rate (%)
threshold = 70
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]


shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)

# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)

print("\n=== FINAL RESULTS ===")

print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")

print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")

print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")


Target:        ['august']
Whisper Tiny:  ['oh', 'baby']  (Time: 1.666s)
Whisper Base:  ['oh', 'bless']  (Time: 1.082s)
Whisper Small: ['oh', 'good']  (Time: 9.420s)



--- TINY REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- BASE REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
Scores -> Tiny: 0 | Base: 0 | Small: 0 



__________________________________________________________________
Target:        ['may']
Whisper Tiny:  ['me']  (Time: 1.540s)
Whisper Base:  ['may']  (Time: 1.084s)
Whisper Small: ['me']  (Time: 2.981s)



--- TINY REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
--- BASE REPORT ---
may             | match
_____________________________________
--- SMALL REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
Sc

# Final Results and Model Comparison

The table below summarizes the performance of all tested models. We evaluated them based on raw accuracy, the impact of the VAD filter, and the performance shift when moving to Beam Size 8.

| Model | Accuracy | Avg Accuracy | Latency | VAD Accuracy | Avg VAD Accuracy | VAD Latency | Beam 8 Accuracy | Avg Beam 8 Accuracy | Beam 8 Latency | Model Size |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **US Model** | 63.58% | 71.90% | 1.9228s | --- | --- | --- | --- | --- | --- | 2.9 GB |
| **Small Model** | 55.63% | 68.28% | 0.3348s | --- | --- | --- | --- | --- | --- | 70.9 MB |
| **Whisper Tiny** | 69.54% | 77.95% | 1.2298s | 68.21% | 76.93% | 0.8192s | 66.89% | 76.93% | 0.7825s | 80 MB |
| **Whisper Base** | 84.77% | 88.05% | 1.9718s |  84.77% | 87.03% | 1.3802s | 84.11% | 86.87% | 1.3200s | 150 MB|
| **Whisper Small** | **87.42%** | **90.05%** | 5.2835s | **86.75%** | **89.19%** | 3.8742s | **86.75%** | **89.32%** | 3.8073s | 490 MB |



---

## Final Conclusion 

### The Winner: Whisper Small

**Whisper Small** is the most accurate model for transcribing non-native children's speech, achieving an `Overall Accuracy of 86.75%` and an `Average Accuracy of 89.19%` when utilizing a **VAD Filter** and **Beam Size 5**. While the pre-tuned model reached a slightly higher `Average Accuracy of 90.05%`, it resulted in the highest latency `~5.2s`. In contrast, our chosen configuration delivers a lower latency `~3.9s` with only a  **1% difference** in accuracy.


**Final Decision:** We will proceed with `Faster-Whisper Small` utilizing a `VAD Filter` and `Beam Size 5` to provide the highest quality results for the end-user.